# Explicit Trapezoidal Runge–Kutta Method (Heun’s Method)

从积分形式出发，梯形求积使用区间左右端速度平均：

$$x_{n+1}=x_n+\frac{h}{2}(v_{left}+v_{right}).$$

右端状态未知，因此先用 Euler 预测：

$$k_1=v_\theta(x_n,t_n),\quad \tilde{x}_{n+1}=x_n+h k_1,$$

$$k_2=v_\theta(\tilde{x}_{n+1},t_n+h),$$

$$x_{n+1}=x_n+\frac{h}{2}(k_1+k_2).$$

Explicit Trapezoidal Runge–Kutta Method 是二阶、2 NFE/步；预测值只用于得到右端速度，不直接作为输出。


## 梯形公式、预测器与校正器

若能取得真轨迹左右端速度，梯形求积给出

$$
\int_{t_n}^{t_n+h}f(x(t),t)dt
=\frac h2\left[f_n+f(x(t_n+h),t_n+h)\right]+O(h^3).
$$

右端真状态未知，先以 Euler 构造预测器

$$
\widetilde x_{n+1}=x_n+h f_n.
$$

由于 $\widetilde x_{n+1}=x(t_n+h)+O(h^2)$，将它放入平滑的右端速度只引入 $O(h^2)$ 的速度误差；乘上 $h$ 后仍为 $O(h^3)$ 的单步误差。于是

$$
k_1=f(x_n,t_n),\quad
k_2=f(x_n+h k_1,t_n+h),\quad
x_{n+1}=x_n+\frac h2(k_1+k_2).
$$

它与中点法同为二阶、同为 2 NFE，但采样节点不同。对某一具体神经速度场，谁更准确不能仅由阶数判断，应在固定 NFE 下测量。


In [ ]:
import torch


class ExplicitTrapezoidalRungeKuttaMethod:
    """Heun method: Euler predictor followed by trapezoidal correction."""

    def __init__(self, velocity):
        self.velocity = velocity
        self.nfe = 0

    @torch.no_grad()
    def sample(self, x_init, times):
        x = x_init
        self.nfe = 0
        for t_now, t_next in zip(times[:-1], times[1:]):
            step = t_next - t_now
            k_left = self.velocity(x, t_now)
            x_right_predictor = x + step * k_left
            k_right = self.velocity(x_right_predictor, t_next)
            x = x + 0.5 * step * (k_left + k_right)
            self.nfe += 2
        return x


times = torch.linspace(0.0, 1.0, 5)
solver = ExplicitTrapezoidalRungeKuttaMethod(lambda x, t: torch.full_like(x, 2.0))
output = solver.sample(torch.zeros(1, 1), times)
torch.testing.assert_close(output, torch.full_like(output, 2.0))
assert solver.nfe == 8
print("constant-velocity invariant passed; NFE =", solver.nfe)
